# 🏦 Pandas pour auditeurs — Exercices Niveau 2 : Moyen

**Contexte** : Vous êtes auditeur en charge de la revue du **portefeuille de prêts immobiliers** d'une banque régionale.
Le service comptable vous a transmis un export de toutes les **échéances** du premier semestre 2024 :
pour chaque prêt, on dispose du montant dû, du montant effectivement encaissé, du statut de paiement et du nombre de jours de retard.

**Objectif** : identifier les agences et les types de prêt à risque, mesurer l'exposition aux impayés,
et vérifier la qualité des données.

**Compétences couvertes** : filtrage multi-conditions (`&`, `|`, `~`, `.isin()`, `.between()`),
colonnes calculées (`np.where`), `groupby` + `.agg()`, `pivot_table`, qualité des données.

> ℹ️ Les données sont entièrement fictives et générées aléatoirement.

## 0. Génération des données — exécutez en premier

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
np.random.seed(2025)

n = 350

agences = ['Agence Nord', 'Agence Sud', 'Agence Est', 'Agence Ouest', 'Agence Centre']
types_pret = ['Immobilier résidentiel', 'Immobilier professionnel', 'Investissement locatif']
statuts = ['Payé', 'Retard léger', 'Retard grave', 'Impayé']

statut_paiement = np.random.choice(statuts, n, p=[0.68, 0.15, 0.10, 0.07])

montants_echeance = np.round(np.random.uniform(500, 4500, n), 2)

# montant payé : 100% si payé, partiel sinon
taux_paiement = np.where(
    statut_paiement == 'Payé', 1.0,
    np.where(statut_paiement == 'Retard léger',
             np.random.uniform(0.0, 1.0, n),
             np.where(statut_paiement == 'Retard grave',
                      np.random.uniform(0.0, 0.5, n),
                      0.0))
)
montants_payes = np.round(montants_echeance * taux_paiement, 2)

jours_retard = np.where(
    statut_paiement == 'Payé', 0,
    np.where(statut_paiement == 'Retard léger', np.random.randint(1, 30, n),
    np.where(statut_paiement == 'Retard grave', np.random.randint(30, 90, n),
             np.random.randint(90, 365, n)))
)

prets = pd.DataFrame({
    'pret_id':          [f'PR{str(i).zfill(5)}' for i in range(1, n + 1)],
    'client_id':        np.random.randint(50000, 51000, n),
    'agence':           np.random.choice(agences, n),
    'type_pret':        np.random.choice(types_pret, n, p=[0.55, 0.25, 0.20]),
    'taux_interet':     np.round(np.random.uniform(1.5, 4.5, n), 2),
    'date_echeance':    pd.to_datetime('2024-01-01') + pd.to_timedelta(
                            np.random.randint(0, 180, n), unit='D'),
    'montant_echeance': montants_echeance,
    'montant_paye':     montants_payes,
    'statut_paiement':  statut_paiement,
    'nb_jours_retard':  jours_retard,
})

# valeurs manquantes volontaires
prets.loc[np.random.choice(prets.index, 10, replace=False), 'taux_interet'] = np.nan
prets.loc[np.random.choice(prets.index, 5,  replace=False), 'montant_paye'] = np.nan

# doublons (même prêt saisi deux fois)
dups = prets.sample(4, random_state=99).copy()
dups['pret_id'] = [f'DUP{i}' for i in range(4)]
prets = pd.concat([prets, dups], ignore_index=True)
prets = prets.sample(frac=1, random_state=3).reset_index(drop=True)

print('Jeu de données prêts prêt :', prets.shape[0], 'échéances,', prets.shape[1], 'colonnes')
prets.head()

**Description des colonnes**

| Colonne | Description |
|---|---|
| `pret_id` | identifiant du prêt |
| `client_id` | identifiant client |
| `agence` | agence gestionnaire |
| `type_pret` | catégorie de prêt |
| `taux_interet` | taux annuel (%) |
| `date_echeance` | date de l'échéance |
| `montant_echeance` | montant dû (€) |
| `montant_paye` | montant effectivement encaissé (€) |
| `statut_paiement` | Payé / Retard léger / Retard grave / Impayé |
| `nb_jours_retard` | nombre de jours de retard (0 si Payé) |

---
## Exercice 1 — Filtrage multi-conditions

**Questions :**
1. Listez les échéances en **`Retard grave`** ou en **`Impayé`** — combien y en a-t-il ?
2. Parmi les `Impayé`s, quelles sont celles dont le **montant dû dépasse 3 000 €** ?
3. Listez les échéances dont le retard est compris **entre 30 et 90 jours** (utilisez `.between()`).
4. Listez les prêts de type `Immobilier professionnel` OU `Investissement locatif`
   dont le statut **n'est pas** `Payé` (utilisez `.isin()` et `~`).

In [ ]:
# 1. Retard grave ou Impayé
# Votre code ici


In [ ]:
# 2. Impayés avec montant > 3 000
# Votre code ici


In [ ]:
# 3. Retard entre 30 et 90 jours
# Votre code ici


In [ ]:
# 4. Prêts pro/locatif NON payés
# Votre code ici


---
## Exercice 2 — Colonnes calculées

**Questions :**
1. Créez une colonne **`montant_impaye`** = `montant_echeance` − `montant_paye`
   (traiter les `NaN` de `montant_paye` comme si le montant payé était 0 — utilisez `.fillna(0)`).
2. Créez une colonne booléenne **`retard_critique`** qui vaut `True` si `nb_jours_retard >= 60`
   (utilisez une comparaison directe).
3. Créez une colonne **`categorie_retard`** avec trois valeurs :
   - `'Aucun'` si `nb_jours_retard == 0`
   - `'Léger'` si `nb_jours_retard` est entre 1 et 59
   - `'Critique'` si `nb_jours_retard >= 60`

   (Astuce : `np.select([condition1, condition2], [choix1, choix2], default=choix3)`)

In [ ]:
# 1. Montant impayé
# Votre code ici


In [ ]:
# 2. Retard critique (booléen)
# Votre code ici


In [ ]:
# 3. Catégorie de retard (np.select)
# Votre code ici


---
## Exercice 3 — Synthèses avec `groupby`

**Questions :**
1. Pour chaque **agence**, calculez : le montant total dû, le montant total payé,
   et le montant total impayé. Triez par montant impayé décroissant.
   Quelle agence présente l'exposition la plus élevée ?
2. Pour chaque **type de prêt**, calculez : le nombre d'échéances, le taux de retard moyen,
   et le montant impayé total.
3. Pour chaque **agence**, calculez le **nombre** d'échéances par `statut_paiement`
   (groupby sur deux colonnes).

In [ ]:
# 1. Synthèse par agence
# Votre code ici


In [ ]:
# 2. Synthèse par type de prêt
# Votre code ici


In [ ]:
# 3. Nombre d'échéances par agence et par statut
# Votre code ici


---
## Exercice 4 — Tableau croisé (`pivot_table`)

**Questions :**
1. Créez un tableau croisé avec les **agences en lignes**, les **types de prêt en colonnes**,
   et la **somme du montant impayé** dans les cases.
2. Créez un second tableau croisé avec les **agences en lignes**, les **statuts en colonnes**,
   et le **nombre d'échéances** dans les cases.
   (Astuce : `aggfunc='count'`, `values='pret_id'`)

In [ ]:
# 1. Pivot : montant impayé par agence × type de prêt
# Votre code ici


In [ ]:
# 2. Pivot : nombre d'échéances par agence × statut
# Votre code ici


---
## Exercice 5 — Qualité des données

**Questions :**
1. Combien de **valeurs manquantes** y a-t-il par colonne ?
2. Identifiez les lignes **en doublon** en vous basant sur les colonnes `client_id`,
   `date_echeance`, `montant_echeance`. Combien de lignes sont concernées ?
3. Créez une version **nettoyée** du DataFrame : sans doublons (garder la première occurrence)
   et avec les `NaN` de `montant_paye` remplacés par 0.
   Combien de lignes restent-il ?

In [ ]:
# 1. Valeurs manquantes par colonne
# Votre code ici


In [ ]:
# 2. Doublons métier
# Votre code ici


In [ ]:
# 3. DataFrame nettoyé
# Votre code ici


---
## Exercice 6 — Analyse de synthèse : agences à risque

Le comité d'audit vous demande un **classement des agences** selon leur taux d'impayé.

**Construisez une table avec, pour chaque agence :**
- Le nombre total d'échéances
- Le nombre d'échéances en `Retard grave` ou `Impayé`
- Le **taux de défaillance** = (retards graves + impayés) / total, en %
- Le montant impayé total

Triez par taux de défaillance décroissant.

(Astuce : calculez les agrégats séparément via `groupby`, puis combinez avec `merge` ou créez les colonnes une par une.)

In [ ]:
# Analyse agences à risque
# Votre code ici
